# Bundle Migration: Export & Transform (Modular)

## Overview

Export and transform dashboards using centralized configuration and helper modules.

**What this notebook does:**
1. Loads config from `config/config.yaml`
2. Discovers dashboards from source workspace
3. Exports dashboard JSONs and permissions
4. Applies catalog/schema transformations
5. Saves transformed files for bundle generation

**Next step:** Run `Bundle_02_Generate_and_Deploy.ipynb`

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml --quiet
dbutils.library.restartPython()

## Cell 1: Import Helpers & Load Config

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd

# Add helpers to path
sys.path.insert(0, '../helpers')

from helpers import (
    load_config,
    get_path,
    create_workspace_client,
    discover_dashboards,
    export_dashboard,
    get_dashboard_permissions,
    load_mapping_csv,
    transform_dashboard_json,
    write_volume_file,
    ensure_directory_exists
)

print("✅ Helper modules imported")

## Cell 2: Load Configuration

In [ ]:
print("="*80)
print("LOADING CONFIGURATION")
print("="*80)

# Load configuration
config = load_config('../config/config.yaml')

print("\n✅ Configuration loaded\n")
print(f"   Source: {config['source']['workspace_url']}")
print(f"   Volume base: {config['paths']['volume_base']}")
print(f"   Discovery method: {config['dashboard_selection']['method']}")

# Ensure directories exist
export_path = get_path('exported')
transformed_path = get_path('transformed')

print("\n📁 Ensuring directories exist...")
ensure_directory_exists(export_path)
ensure_directory_exists(transformed_path)
print(f"   ✅ Export: {export_path}")
print(f"   ✅ Transformed: {transformed_path}")

## Cell 3: Discover Dashboards

In [ ]:
print("\n" + "="*80)
print("DISCOVERING DASHBOARDS")
print("="*80)

# Connect to source workspace
print("\n🔗 Connecting to source workspace...")
source_client = create_workspace_client('source')
print(f"   ✅ Connected\n")

# Discover dashboards
print(f"🔍 Discovering dashboards using: {config['dashboard_selection']['method']}")
dashboards = discover_dashboards(source_client)

print(f"\n✅ Found {len(dashboards)} dashboard(s)\n")

if dashboards:
    df = pd.DataFrame([
        {
            'ID': d['id'],
            'Name': d['name'],
            'Path': d.get('path', 'N/A')
        }
        for d in dashboards
    ])
    display(df)
else:
    print("⚠️  No dashboards found. Check your configuration.")

## Cell 4: Export Dashboards & Permissions

In [ ]:
print("\n" + "="*80)
print("EXPORTING DASHBOARDS & PERMISSIONS")
print("="*80)

if not dashboards:
    print("\n⚠️  No dashboards to export")
else:
    export_results = []
    
    for i, dash in enumerate(dashboards, 1):
        dashboard_id = dash['id']
        print(f"\n[{i}/{len(dashboards)}] Exporting: {dash['name']}")
        
        try:
            # Export dashboard JSON
            json_content, display_name, clean_name = export_dashboard(source_client, dashboard_id)
            
            # Save JSON
            json_file = f"{export_path}/dashboard_{dashboard_id}_{clean_name}.lvdash.json"
            write_volume_file(json_file, json_content)
            print(f"   ✅ Exported JSON")
            
            # Get and save permissions
            permissions = get_dashboard_permissions(source_client, dashboard_id)
            permissions['display_name'] = display_name
            
            perm_file = f"{export_path}/dashboard_{dashboard_id}_{clean_name}_permissions.json"
            write_volume_file(perm_file, json.dumps(permissions, indent=2))
            
            acl_count = len(permissions.get('access_control_list', []))
            print(f"   🔐 Permissions: {acl_count} ACL(s)")
            
            export_results.append({
                'dashboard_id': dashboard_id,
                'name': display_name,
                'status': 'success',
                'json_file': json_file
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            export_results.append({
                'dashboard_id': dashboard_id,
                'name': dash['name'],
                'status': 'failed'
            })
    
    successful = len([r for r in export_results if r['status'] == 'success'])
    print(f"\n✅ Successfully exported: {successful}/{len(dashboards)}")

## Cell 5: Transform Dashboards

In [ ]:
print("\n" + "="*80)
print("TRANSFORMING DASHBOARDS")
print("="*80)

if not config['transformation']['enabled']:
    print("\n⚠️  Transformation disabled in config")
elif not export_results or successful == 0:
    print("\n⚠️  No dashboards to transform")
else:
    # Load mapping CSV
    mapping_csv_path = get_path('volume_base') + "/" + config['transformation']['mapping_csv']
    print(f"\n📋 Loading mappings: {mapping_csv_path}")
    
    try:
        mappings = load_mapping_csv(mapping_csv_path)
        print(f"   ✅ Loaded {len(mappings)} mapping rule(s)\n")
        
    except Exception as e:
        print(f"   ❌ Failed to load CSV: {e}")
        raise
    
    # Transform each dashboard
    transform_results = []
    successful_exports = [r for r in export_results if r['status'] == 'success']
    
    for i, result in enumerate(successful_exports, 1):
        name = result['name']
        json_file = result['json_file']
        
        print(f"[{i}/{len(successful_exports)}] Transforming: {name}")
        
        try:
            # Read exported JSON
            from helpers.volume_utils import read_volume_file
            json_content = read_volume_file(json_file)
            
            # Apply transformations
            transformed = transform_dashboard_json(json_content, mappings)
            
            # Save transformed version
            filename = Path(json_file).name
            transformed_file = f"{transformed_path}/{filename}"
            write_volume_file(transformed_file, transformed)
            
            print(f"   ✅ Transformed")
            
            transform_results.append({
                'dashboard_id': result['dashboard_id'],
                'name': name,
                'status': 'success'
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            transform_results.append({
                'dashboard_id': result['dashboard_id'],
                'name': name,
                'status': 'failed'
            })
    
    successful_transforms = len([r for r in transform_results if r['status'] == 'success'])
    
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"\n✅ Exported: {successful}/{len(dashboards)}")
    print(f"✅ Transformed: {successful_transforms}/{successful}")
    print(f"\n📁 Files ready at: {transformed_path}")
    print(f"\n▶️  Next: Run Bundle_02_Generate_and_Deploy.ipynb")